In [1]:
import polars as pl
import polars_sha as sha
from hashlib import sha256, sha512

In [4]:
df = pl.scan_parquet("data/2019-Oct-10M.parquet")

In [5]:
df.head().collect()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,i32,i64,str,str,f32,i32,str
"""2019-10-01 00:00:00 UTC""","""view""",44600062,2103807459595387724,null,"""shiseido""",35.790001,541312140,"""72d76fde-8bb3-4e00-8c23-a032df…"
"""2019-10-01 00:00:00 UTC""","""view""",3900821,2053013552326770905,"""appliances.environment.water_h…","""aqua""",33.200001,554748717,"""9333dfbd-b87a-4708-9857-633655…"
"""2019-10-01 00:00:01 UTC""","""view""",17200506,2053013559792632471,"""furniture.living_room.sofa""",null,543.099976,519107250,"""566511c2-e2e3-422b-b695-cf8e6e…"
"""2019-10-01 00:00:01 UTC""","""view""",1307067,2053013558920217191,"""computers.notebook""","""lenovo""",251.740005,550050854,"""7c90fc70-0e80-4590-96f3-13c02c…"
"""2019-10-01 00:00:04 UTC""","""view""",1004237,2053013555631882655,"""electronics.smartphone""","""apple""",1081.97998,535871217,"""c6bd7419-2748-4c56-95b4-8cec9f…"


In [6]:
df_udf = df.with_columns(
    pl.col("category_code").map_elements(lambda x: sha256(x.encode()).hexdigest(), return_dtype=pl.Utf8).alias("category_code_sha256"),
    pl.col("category_code").map_elements(lambda x: sha512(x.encode()).hexdigest(), return_dtype=pl.Utf8).alias("category_code_sha512"),
).sort("category_code_sha256", nulls_last=True).collect()

In [7]:
df_udf.head()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_code_sha256,category_code_sha512
str,str,i32,i64,str,str,f32,i32,str,str,str
"""2019-10-01 02:18:07 UTC""","""view""",1500259,2053013552955916539,"""computers.peripherals.printer""","""epson""",154.179993,534516750,"""09e710f2-94c7-491d-b56a-2fbc5c…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:18:53 UTC""","""view""",1500451,2053013552955916539,"""computers.peripherals.printer""",null,122.900002,512757140,"""0938b9e1-c6b7-46b7-93d9-1e9067…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:19:29 UTC""","""view""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:21:04 UTC""","""cart""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:21:33 UTC""","""cart""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"


In [8]:
del df_udf

In [9]:
df_native = df.with_columns(
    sha.sha2_256("category_code").alias('category_code_sha256'),
    sha.sha2_512("category_code").alias('category_code_sha512')
).sort("category_code_sha256", nulls_last=True).collect()

In [10]:
df_native.head()

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_code_sha256,category_code_sha512
str,str,i32,i64,str,str,f32,i32,str,str,str
"""2019-10-01 02:18:07 UTC""","""view""",1500259,2053013552955916539,"""computers.peripherals.printer""","""epson""",154.179993,534516750,"""09e710f2-94c7-491d-b56a-2fbc5c…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:18:53 UTC""","""view""",1500451,2053013552955916539,"""computers.peripherals.printer""",null,122.900002,512757140,"""0938b9e1-c6b7-46b7-93d9-1e9067…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:19:29 UTC""","""view""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:21:04 UTC""","""cart""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
"""2019-10-01 02:21:33 UTC""","""cart""",1500021,2053013552955916539,"""computers.peripherals.printer""","""epson""",116.730003,513489079,"""63de1120-9d48-45d0-b024-500d1b…","""00e4e9e0399a119dc2d7304e2da781…","""dda8faed21d06e10d0e1385aa46cff…"
